# Promptfoo: 自定义代码评分器

**注意：这节课位于包含相关代码文件的文件夹中。如果你想要跟随教程并自己运行评估，请下载整个文件夹**


到目前为止，我们已经学习了如何使用 promptfoo 内置的一些评分器，如 `exact-match` 和 `contains-all`。这些功能通常很有用，但 promptfoo 也允许我们编写自定义评分逻辑来处理更具体的评分任务。

为了演示这一点，我们将使用一个非常简单的提示词模板：

> 写一段关于 {{topic}} 的短文。确保你恰好提到 {{topic}} {{count}} 次，不多不少。

我们将用 `"tweezers"` 和 `7` 这样的值填充 `{{topic}}` 和 `{{count}}`，得到如下提示词：

> 写一段关于 tweezers 的短文。确保你恰好提到 tweezers 7 次，不多不少。

为了对这个输出进行评分，我们需要编写一些自定义逻辑来确保模型的输出恰好提到 "tweezers" 7 次。

对于以下提示词：

> 写一段关于 sheep 的短文。确保你恰好提到 sheep 3 次，不多不少。

我们需要编写评分逻辑来确保单词 "sheep" 在模型输出中恰好出现 3 次。

---

## 初始化 promptfoo

一如既往，第一步是使用以下命令初始化 promptfoo：


```bash
npx promptfoo@latest init
```


如前所述，这会创建一个 `promptfooconfig.yaml` 文件。我们可以删除现有内容。

接下来，我们将配置我们的 providers。在 `promptfooconfig.yaml` 中添加以下内容：

```yaml
description: Count mentions

providers:
  - anthropic:messages:claude-3-haiku-20240307
  - anthropic:messages:claude-3-5-sonnet-20240620
```
这告诉 promptfoo，我们希望使用 Claude 3 Haiku 和 Claude 3.5 Sonnet 来运行评估。我们将比较它们在这个特定任务上的表现！

确保已设置 `ANTHROPIC_API_KEY` 环境变量。你可以通过在终端中运行以下命令来设置环境变量：

```bash
export ANTHROPIC_API_KEY=your_api_key_here
```

---

## 准备提示词

到目前为止，我们已经看到可以将提示词作为 Python 文件中的函数来编写。这是我们推荐的方法，但 promptfoo 提供了几种其他选项来指定提示词。最简单的选项是将提示词直接写在 YAML 文件中。

让我们尝试这种内联方法。更新 `promptfooconfig.yaml` 文件，添加以下内容：


```yaml
description: Count mentions
prompts:
  - >-
    Write a short paragraph about {{topic}}. Make sure you mention {{topic}} exactly {{count}} times, no more or fewer. Only use lower case letters in your output.
providers:
  - anthropic:messages:claude-3-haiku-20240307
  - anthropic:messages:claude-3-5-sonnet-20240620
```


注意 `prompts` 字段，它将文本提示词直接包含在 YAML 文件中。请注意使用双花括号的 `{{topic}}` 和 `{{count}}` 变量。这些提示词使用 Nunjucks 模板语法，这一点很重要！

---

## 编写测试用例

在之前的课程中，我们将测试用例和评分逻辑写在 CSV 文件中。如前所述，promptfoo 非常灵活，提供了多种指定测试的方法。

我们可以直接在 YAML 配置文件中编写测试用例。更新 `promptfooconfig.yaml` 文件，如下所示：

```yaml
description: Count mentions
prompts:
  - >-
    Write a short paragraph about {{topic}}. Make sure you mention {{topic}} exactly {{count}} times, no more or fewer. Only use lower case letters in your output.
providers:
  - anthropic:messages:claude-3-haiku-20240307
  - anthropic:messages:claude-3-5-sonnet-20240620
tests:
  - vars:
      topic: sheep
      count: 3
  - vars:
      topic: fowl
      count: 2
  - vars:
      topic: gallows
      count: 4
  - vars:
      topic: tweezers
      count: 7
  - vars:
      topic: jeans
      count: 6
```

在底部，我们定义了 5 个测试用例，每个用例都有自己 的 `topic` 和 `count` 值。Promptfoo 会自动运行每个测试，替换提示词模板中的 `{{topic}}` 和 `{{count}}`。

我们还没有任何评分逻辑，但仍然可以运行评估来确保我们的变量正确添加。

要运行评估，我们将使用之前见过的相同命令：

```bash
npx promptfoo@latest eval
```

这是我们得到的输出：



如果我们放大单行，可以看到模型输出总体看起来不错。在这个例子中，`{{topic}}` 被设置为 "sheep"，对应的模型输出是关于 sheep 的段落！



现在我们只需要实现自定义评分逻辑来测试输出是否恰好以正确的次数提到了主题！

---

## 添加自定义评分器函数

Promptfoo 允许我们定义自己的 Python 评分器函数。对于这个特定的例子，我们想要定义一个函数来确保模型输出以正确的次数提到特定的主题。我们首先定义一个新的 Python 文件 `count.py`。在这个文件中，我们将添加以下函数：

```py
import re

def get_assert(output, context):
    topic = context["vars"]["topic"]
    goal_count = int(context["vars"]["count"])
    pattern = fr'(^|\s)\b{re.escape(topic)}\b'

    actual_count = len(re.findall(pattern, output.lower()))

    pass_result = goal_count == actual_count

    result = {
        "pass": pass_result,
        "score": 1 if pass_result else 0,
        "reason": f"Expected {topic} to appear {goal_count} times. Actual: {actual_count}",
    }
    return result
```

让我们讨论一下上面的代码做了什么。Promptfoo 会自动在我们的文件中查找名为 `get_assert` 的函数。它会传递两个参数：

- 来自给定模型的输出
- `context` 字典，其中包含生成输出的变量和提示词

Promptfoo 期望我们的函数返回以下之一：
- 一个布尔值（通过/失败）
- 一个浮点数（分数）
- 一个 GradingResult 字典

我们选择返回 GradingResult 字典，它必须包含以下属性：

- `pass_`: 布尔值
-  `score`: 浮点数
- `reason`: 字符串说明

在上面的函数中，我们从 `context` 参数中提取主题和计数，然后使用正则表达式计算主题在输出中出现的次数，最后返回 `result`

现在我们已经定义好了评分器，是时候告诉 promptfoo 关于它了。更新 `promptfooconfig.yaml` 文件：

```yaml
description: Count mentions
prompts:
  - >-
    Write a short paragraph about {{topic}}. Make sure you mention {{topic}} exactly {{count}} times, no more or fewer. Only use lower case letters in your output.
providers:
  - anthropic:messages:claude-3-haiku-20240307
  - anthropic:messages:claude-3-5-sonnet-20240620
defaultTest:
  assert:
    - type: python
      value: file://count.py
tests:
  - vars:
      topic: sheep
      count: 3
  - vars:
      topic: fowl
      count: 2
  - vars:
      topic: gallows
      count: 4
  - vars:
      topic: tweezers
      count: 7
  - vars:
      topic: jeans
      count: 6
```
`defaultTest` 告诉 promptfoo，对于它运行的每个测试，我们都希望使用在 `count.py` 文件中定义的 Python 评分器。

---

## 运行评估

要运行评估，我们将使用之前见过的相同命令：

```bash
npx promptfoo@latest eval
```

这是我们运行评估后得到的输出：



运行此命令启动 Web 界面：

```bash
npx promptfoo@latest view
```



我们可以看到 Claude 3.5 在这个任务上得了 100%，而 Claude 3 Haiku 得了 20%。要验证结果，请点击放大镜图标查看完整的输入提示词和相应的输出。

以下是 Claude 3 Haiku 的错误输出：



以及 Claude 3.5 Sonnet 的正确输出：



这个特定的评估有点傻，但它的目的是演示定义自定义 Python 评分器逻辑的过程。通过内置的 promptfoo 断言和自定义评分器函数，我们可以编写几乎任何代码评分评估。

在下一节课中，我们将学习 promptfoo 中的模型评分评估。